In [1]:
# https://www.nature.com/articles/s41597-024-03378-x#Sec11
# file:///C:/Users/2D1245897/Downloads/s41597-024-03378-x.pdf
# https://zenodo.org/records/11085013

In [2]:
import os
os.environ['USE_PYGEOS'] = '0'

import numpy
import pandas
from datetime import datetime, timedelta
import pytz
import geopandas
import dask_geopandas
import shapely
from shapely import wkb, wkt
import pyproj

import subprocess
import shutil
from glob import glob
import json

from functools import partial
from multiprocessing import Pool
from random import shuffle

import matplotlib.pyplot as plt

In [3]:
raw_geoparquet_path = '/data/vector/rooftop_area_growth_projections/raw/11085013/Dataset_V2_29042024/Vector'
processed_geoparquet_path = '/data/vector/rooftop_area_growth_projections/processed'
files = glob(os.path.join(raw_geoparquet_path, '*.gpkg'))
len(files)
files

['/data/vector/rooftop_area_growth_projections/raw/11085013/Dataset_V2_29042024/Vector\\Results_Vis.gpkg']

In [4]:
gdf = geopandas.read_file(files[0])
gdf.tail(2)

,FN_ID,CF,BF_1_20,BF_1_30,BF_1_40,BF_1_50,BF_2_20,BF_2_30,BF_2_40,BF_2_50,...,BF_4_20,BF_4_30,BF_4_40,BF_4_50,BF_5_20,BF_5_30,BF_5_40,BF_5_50,Check,geometry
618757,3070966,0.1459,0.0121,0.0121,0.0121,0.0121,0.0121,0.0121,0.0121,0.0121,...,0.0121,0.0121,0.0121,0.0121,0.0121,0.0121,0.0121,0.0121,0.242,"MULTIPOLYGON (((-69.37500 77.37500, -69.37500 ..."
618758,3070967,0.1459,0.0226,0.0226,0.0226,0.0226,0.0226,0.0226,0.0226,0.0226,...,0.0226,0.0226,0.0226,0.0226,0.0226,0.0226,0.0226,0.0226,0.452,"MULTIPOLYGON (((-69.25000 77.37500, -69.25000 ..."


In [5]:
# Stack and create scenario and time columns
gdf = gdf.set_index(['FN_ID', 'CF', 'Check', 'geometry']).stack().reset_index()
gdf = gdf.rename(columns={0: 'rooftop area'})
gdf[['dummy', 'scenario', 'time']] = pandas.DataFrame(gdf['level_4'].str.split('_').values.tolist())
del gdf['dummy']
gdf['scenario'] = gdf['scenario'].apply(lambda x: 'SSP'+x)
gdf['time'] = gdf['time'].apply(lambda x: datetime(int(x)+2000, 1, 1, tzinfo=pytz.utc))
gdf = gdf.rename(columns={'level_4': 'BF_X_Y'})

gdf = geopandas.GeoDataFrame(gdf)
# Translate Multipolygon to Polygon
assert(set(gdf['geometry'].apply(lambda x: len(x.geoms)))=={1})
gdf['geometry'] = gdf['geometry'].apply(lambda x: x.geoms[0])

cols = ['FN_ID', 'CF', 'Check', 'BF_X_Y', 'scenario', 'time', 'rooftop area', 'geometry']
gdf = gdf[cols]
gdf

,FN_ID,CF,Check,BF_X_Y,scenario,time,rooftop area,geometry
0,18187,0.1459,0.0658,BF_1_20,SSP1,2020-01-01 00:00:00+00:00,0.0000,"POLYGON ((-66.75000 -55.12500, -66.75000 -55.0..."
1,18187,0.1459,0.0658,BF_1_30,SSP1,2030-01-01 00:00:00+00:00,0.0000,"POLYGON ((-66.75000 -55.12500, -66.75000 -55.0..."
2,18187,0.1459,0.0658,BF_1_40,SSP1,2040-01-01 00:00:00+00:00,0.0000,"POLYGON ((-66.75000 -55.12500, -66.75000 -55.0..."
3,18187,0.1459,0.0658,BF_1_50,SSP1,2050-01-01 00:00:00+00:00,0.0000,"POLYGON ((-66.75000 -55.12500, -66.75000 -55.0..."
4,18187,0.1459,0.0658,BF_2_20,SSP2,2020-01-01 00:00:00+00:00,0.0000,"POLYGON ((-66.75000 -55.12500, -66.75000 -55.0..."
...,...,...,...,...,...,...,...,...
12375175,3070967,0.1459,0.4520,BF_4_50,SSP4,2050-01-01 00:00:00+00:00,0.0226,"POLYGON ((-69.25000 77.37500, -69.25000 77.500..."
12375176,3070967,0.1459,0.4520,BF_5_20,SSP5,2020-01-01 00:00:00+00:00,0.0226,"POLYGON ((-69.25000 77.37500, -69.25000 77.500..."
12375177,3070967,0.1459,0.4520,BF_5_30,SSP5,2030-01-01 00:00:00+00:00,0.0226,"POLYGON ((-69.25000 77.37500, -69.25000 77.500..."
12375178,3070967,0.1459,0.4520,BF_5_40,SSP5,2040-01-01 00:00:00+00:00,0.0226,"POLYGON ((-69.25000 77.37500, -69.25000 77.500..."


In [10]:
gdf_plot = gdf[(gdf['scenario']=='SSP1') & (gdf['time']==datetime(2050,1,1,tzinfo=pytz.utc))]
gdf_plot = gdf_plot[gdf_plot['rooftop area']!=0].reset_index(drop=True)
gdf_plot.tail(2)

,FN_ID,CF,Check,BF_X_Y,scenario,time,rooftop area,geometry
454656,3070966,0.1459,0.242,BF_1_50,SSP1,2050-01-01 00:00:00+00:00,0.0121,"POLYGON ((-69.37500 77.37500, -69.37500 77.500..."
454657,3070967,0.1459,0.452,BF_1_50,SSP1,2050-01-01 00:00:00+00:00,0.0226,"POLYGON ((-69.25000 77.37500, -69.25000 77.500..."


In [ ]:
gdf_plot.plot(numpy.log10(gdf_plot['rooftop area']), cmap='coolwarm', figsize=(20,8), legend=True)
plt.title('Log10(rooftop area)')
plt.show()

In [27]:
out_file = os.path.join(processed_geoparquet_path, 'rooftop_area.parquet')
os.makedirs(processed_geoparquet_path, exist_ok=True)
gdf.to_parquet(out_file)

In [28]:
gdf_map = pandas.read_parquet(
    "/data/vector/rooftop_area_growth_projections/raw/11085013/Dataset_V2_29042024/Models/FN_MAP.parquet",
)
gdf_map['WKT'] = geopandas.GeoSeries.from_wkt(gdf_map['WKT'])
gdf_map = geopandas.GeoDataFrame(gdf_map, geometry='WKT')
gdf_map.T

,0,1,2,3,4,5,6,7,8,9,...,3216950,3216951,3216952,3216953,3216954,3216955,3216956,3216957,3216958,3216959
WKT,"MULTIPOLYGON (((-23.125 -42.75, -23.125 -42.62...","MULTIPOLYGON (((-23 -42.75, -23 -42.625, -22.8...","MULTIPOLYGON (((-22.875 -42.75, -22.875 -42.62...","MULTIPOLYGON (((-22.75 -42.75, -22.75 -42.625,...","MULTIPOLYGON (((-22.625 -42.75, -22.625 -42.62...","MULTIPOLYGON (((-22.5 -42.75, -22.5 -42.625, -...","MULTIPOLYGON (((-22.375 -42.75, -22.375 -42.62...","MULTIPOLYGON (((-22.25 -42.75, -22.25 -42.625,...","MULTIPOLYGON (((-22.125 -42.75, -22.125 -42.62...","MULTIPOLYGON (((-22 -42.75, -22 -42.625, -21.8...",...,"MULTIPOLYGON (((-30.625 -49.625, -30.625 -49.5...","MULTIPOLYGON (((-30.5 -49.625, -30.5 -49.5, -3...","MULTIPOLYGON (((-30.375 -49.625, -30.375 -49.5...","MULTIPOLYGON (((-30.25 -49.625, -30.25 -49.5, ...","MULTIPOLYGON (((-30.125 -49.625, -30.125 -49.5...","MULTIPOLYGON (((-30 -49.625, -30 -49.5, -29.87...","MULTIPOLYGON (((-29.875 -49.625, -29.875 -49.5...","MULTIPOLYGON (((-29.75 -49.625, -29.75 -49.5, ...","MULTIPOLYGON (((-29.625 -49.625, -29.625 -49.5...","MULTIPOLYGON (((-29.5 -49.625, -29.5 -49.5, -2..."
FN_ID,303656,303657,303658,303659,303660,303661,303662,303663,303664,303665,...,145196,145197,145198,145199,145200,145201,145202,145203,145204,145205


In [29]:
df_results = pandas.read_parquet("/data/vector/rooftop_area_growth_projections/raw/11085013/Dataset_V2_29042024/Numerical/Results.parquet")
df_results.T

,0,1,2,3,4,5,6,7,8,9,...,3216950,3216951,3216952,3216953,3216954,3216955,3216956,3216957,3216958,3216959
FN_ID,1.0000,2.0000,3.0000,4.0000,5.0000,6.0000,7.0000,8.0000,9.0000,10.0000,...,3.216951e+06,3.216952e+06,3.216953e+06,3.216954e+06,3.216955e+06,3.216956e+06,3.216957e+06,3.216958e+06,3.216959e+06,3.216960e+06
CF,0.1459,0.1459,0.1459,0.1459,0.1459,0.1459,0.1459,0.1459,0.1459,0.1459,...,1.459000e-01,1.459000e-01,1.459000e-01,1.459000e-01,1.459000e-01,1.459000e-01,1.459000e-01,1.459000e-01,1.459000e-01,1.459000e-01
BF_1_20,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
BF_1_30,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
BF_1_40,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
BF_1_50,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
BF_2_20,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
BF_2_30,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
BF_2_40,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
BF_2_50,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


In [30]:
dask_geopandas.read_parquet(processed_geoparquet_path, split_row_groups=False)

,FN_ID,CF,Check,BF_X_Y,scenario,time,rooftop area,geometry
npartitions=1,,,,,,,,
,int64,float64,float64,object,object,"datetime64[ns, UTC]",float64,geometry
,...,...,...,...,...,...,...,...
